In [ ]:
import pandas as pd  
from scan_spectrolyser import scan_helpers
from scan_spectrolyser import no3_calibrations
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from hampel import hampel
import os


# Data Processing

## Import all FP and Correct Timestamp

In [ ]:
rme = scan_helpers.import_all_scan_fp('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Readings/', correct_dst=True, tz='US/Mountain')
rme = rme['2025-01-23':].dropna(how='all') # only use data after install date
#rme = scan_helpers.remove_invalid_abs(rme, 55)


In [ ]:
rme.loc[rme.loc[:,380.00:700.00].isna().any(axis=1), 380.00:700.00]

## Correct for Turbidity

In [ ]:
rme_corrected = scan_helpers.correct_turbidity(rme)


## Apply Calibrations

## Initial Calibrations

In [ ]:
rme = scan_helpers.apply_calibrations(rme, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})
rme['two_wavelength_no3_mgl'] = (rme['two_wavelength'] * 3.6615-.0802)* 14/1000
rme['one_wavelength_no3_mgl'] = (rme['one_wavelength']*3.7314+.3704) * 14/1000
rme['second_derivative_no3_mgl'] = (rme['second_derivative'] * 42.7379-.2905)* 14/1000

rme_corrected = scan_helpers.apply_calibrations(rme_corrected, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})
rme_corrected['two_wavelength_no3_mgl_correct'] = (rme_corrected['two_wavelength'] * 3.6451-.0861)* 14/1000
rme_corrected['one_wavelength_no3_mgl_correct'] = (rme_corrected['one_wavelength']*3.9753+.4317) * 14/1000
rme_corrected['second_derivative_no3_mgl_correct'] = (rme_corrected['second_derivative'] * 42.7490-.2897)* 14/1000
rme_corrected

## After Feb 6 S::CAN Swap
- moved Dobson S::CAN up to RME, so these are old dobson calibrations

In [ ]:
rme.loc['2/6/2026':, 'two_wavelength_no3_mgl'] = (rme.loc['2/6/2026':]['two_wavelength'] * 3.7649+.1904)* 14/1000
rme.loc['2/6/2026':, 'one_wavelength_no3_mgl'] = (rme.loc['2/6/2026':]['one_wavelength']*3.6962-.6334) * 14/1000
rme.loc['2/6/2026':, 'second_derivative_no3_mgl'] = (rme.loc['2/6/2026':]['second_derivative'] * 43.0111-.9440)* 14/1000



rme_corrected.loc['2/6/2026':, 'two_wavelength_no3_mgl_correct'] = (rme_corrected.loc['2/6/2026':]['two_wavelength'] * 3.6742-.1730)* 14/1000
rme_corrected.loc['2/6/2026':, 'one_wavelength_no3_mgl_correct'] = (rme_corrected.loc['2/6/2026':]['one_wavelength']*4.0634+.1136) * 14/1000
rme_corrected.loc['2/6/2026':, 'second_derivative_no3_mgl_correct'] = (rme_corrected.loc['2/6/2026':]['second_derivative'] * 43.0231-.9425)* 14/1000
rme_corrected[['two_wavelength_no3_mgl', 'one_wavelength_no3_mgl', 'second_derivative_no3_mgl']] = rme[['two_wavelength_no3_mgl', 'one_wavelength_no3_mgl', 'second_derivative_no3_mgl']]
rme[['two_wavelength_no3_mgl_correct', 'one_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct']] = rme_corrected[['two_wavelength_no3_mgl_correct', 'one_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct']]


rme_corrected

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

nitrate_cols = ['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl', 'one_wavelength_no3_mgl_correct', 'two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct']

rme_corrected.plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl','two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#rme.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))
ax.set_ylim((-.1, .8))

## Cleanup Bad Values From Site Visits

In [ ]:
fig, ax= plt.subplots()

rme_corrected[:'2025-01-23 00:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected[:'2025-01-23'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


- loooks like the initail turbidity cleared up around 1-23-25 at 18:00
- second_derivative method  rejected the turbidity, returned to normal almost immediately

In [ ]:
fig, ax= plt.subplots()

rme_corrected['2025-01-30 10:00':'2025-01-30 13:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected['2025-01-30 10:00':'2025-01-30 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- no spike in corrected data

In [ ]:
fig, ax= plt.subplots()

rme_corrected['2025-02-12 10:20':'2025-02-13 00:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected['2025-02-12 10:20':'2025-02-13 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- looks like readings bad from 10:20 to 18:00
- again, second_derivative clears up before two wavelength method and absorbance

In [ ]:
fig, ax= plt.subplots()

rme_corrected['2025-02-25 10:30':'2025-02-25 15:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected['2025-02-25 10:30':'2025-02-25 15:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- data are bad from 2025-02-25 12:00 to 13:30.
- this time 2nd derivative and two_ewavelength recover about the same

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-03-11 08:00':'2025-03-11 15:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-03-11 08:00':'2025-03-11 15:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- data are bad from 9:00 to 13:00

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-03-25 10:00':'2025-03-25 14:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-03-25 10:00':'2025-03-25 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- data bad from 10:10 to 11:30

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-04-02 9:00':'2025-04-02 14:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-04-02 9:00':'2025-04-02 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

data bad from 9:30 to 10:30

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-04-17 10:00':'2025-04-17 16:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-04-17 10:00':'2025-04-17 16:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- no bad data from this field visit

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-05-01 10:00':'2025-05-01 16:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-05-01 10:00':'2025-05-01 16:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- also no bad data here

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-05-12 10:00':'2025-05-12 16:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-05-12 10:00':'2025-05-12 16:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-04-28'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-04-28'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-05-22 12:00':'2025-05-22 13:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-05-22 12:00':'2025-05-22 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-05-30 9:30':'2025-05-30 12:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-05-30 9:30':'2025-05-30 12:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-06-11 11:30':'2025-06-11 13:30'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-06-11 11:30':'2025-06-11 13:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-06-26 11:45':'2025-06-26 14:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-06-26 11:45':'2025-06-26 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-07-07 12:00':'2025-07-08 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-07-07 12:00':'2025-07-08 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-07-23 12:00':'2025-07-24 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-07-23 12:00':'2025-07-24 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-08-05 12:00':'2025-08-06 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-08-05 12:00':'2025-08-06 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-08-19 06:00':'2025-08-20 00:00'].plot(y=['two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-08-19 06:00':'2025-08-20 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-08-19 06:00':'2025-08-20 00:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-08-19 06:00':'2025-08-20 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- looks like absorbance filter mostly caught high absorbances, but drop from 08:00 to 13:00 anyway. 
- weirdly, nitrate seems to spike after visit - is this real from disturbing the sediments or an artifact of turbidity correction?

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-08-29 06:00':'2025-08-29 16:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-08-29 06:00':'2025-08-29 16:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-9-09 06:00':'2025-09-09 16:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-09-09 06:00':'2025-09-09 16:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-09-18 06:00':'2025-09-18 16:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-09-18 06:00':'2025-09-18 16:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-09-30 11:00':'2025-10-01 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-09-30 11:00':'2025-10-01 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-10-16 10:00':'2025-10-16 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-10-16 10:00':'2025-10-16 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-10-31 10:00':'2025-10-31 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-10-31 10:00':'2025-10-31 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-11-20 11:30':'2025-11-20 17:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-11-20 11:30':'2025-11-20 17:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-12-04 10:00':'2025-12-04 18:30'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-12-04 10:00':'2025-12-04 18:30'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2025-12-30 12:30':'2025-12-30 17:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2025-12-30 12:30':'2025-12-30 17:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme.loc['2026-01-14 11:00':'2026-01-14 14:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-01-14 11:00':'2026-01-14 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2026-01-27 12:10:00':'2026-01-27 19:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-01-27 12:15:00':'2026-01-27 19:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2026-02-06 10:30':'2026-02-06 19:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-02-06 10:30':'2026-02-06 19:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2026-02-25 10:30':'2026-02-25 19:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-02-25 10:30':'2026-02-25 19:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2026-03-10 11:00':'2026-03-10 14:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-03-10 11:00':'2026-03-10 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2026-03-23 11:30':'2026-03-23 13:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-03-23 11:30':'2026-03-23 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2026-04-06 09:00':'2026-04-06 13:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-04-06 09:00':'2026-04-06 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2026-04-21 09:00':'2026-04-21 18:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme.loc['2026-04-21 09:00':'2026-04-21 18:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
rme_bad = [
    ('2025-01-23 00:00', '2025-01-23 18:00'),
    ('2025-02-12 10:20', '2025-02-12 18:00'),
    ('2025-02-25 12:00', '2025-02-25 13:30'),
    ('2025-03-11 9:00', '2025-03-11 13:00'),
    ('2025-03-25 10:10', '2025-03-25 11:30'),
    ('2025-04-02 9:30', '2025-04-02 10:30'),
    ('2025-04-28 11:00', '2025-04-28 21:00'),
    ('2025-05-22 12:00','2025-05-22 13:00'),
    ('2025-06-11 11:30','2025-06-11 12:45'),
    ('2025-06-26 11:45','2025-06-26 13:00'),
    ('2025-07-07 12:00','2025-07-08 00:00'),
    ('2025-08-19 08:00','2025-08-19 13:00'),
    ('2025-08-29 10:00','2025-08-29 16:30'),
    ('2025-09-09 10:00','2025-09-09 12:00'),
    ('2025-09-18 9:00', '2025-09-18 12:30'),
    ('2025-09-30 12:00', '2025-10-01 06:00'),
    ('2025-10-16 10:30', '2025-10-16 16:00'),
    ('2025-11-20 11:30', '2025-11-20 17:00'),
    ('2025-12-04 10:00','2025-12-04 17:30'),
    ('2025-12-30 12:30','2025-12-30 17:00'),
    ('2026-01-14 11:00','2026-01-14 14:00'),
    ('2026-01-27 12:15:00','2026-01-27 18:00'),
    ('2026-02-06 10:30','2026-02-06 16:00'),
    ('2026-03-10 12:00','2026-03-10 14:00'),
    ('2026-03-23 11:30','2026-03-23 13:00'),
    
    
    

    
    
]
rme_mask = np.array([
    (rme.index >= start) & (rme.index <= end)
    for start, end in rme_bad
]).any(axis=0)

rme_cleaned = rme_corrected.loc[~rme_mask]
rme_cleaned_uncorrected = rme.loc[~rme_mask]
rme_cleaned



### Replotting time series to see where absorbance threshold exceeded

In [ ]:
fig, ax = plt.subplots(figsize=(11, 11), nrows = 9, sharey=True)

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2025-01-01':'2025-03-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[0], legend=False)
rme_cleaned_uncorrected.loc['2025-01-01':'2025-03-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[0], legend=False)
ax[0].set_xlim(pd.to_datetime('2025-01-01'), pd.to_datetime('2025-03-01'))

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2025-03-01':'2025-05-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[1], legend=False)
rme_cleaned_uncorrected.loc['2025-03-01':'2025-05-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[1], legend=False)
ax[1].set_xlim(pd.to_datetime('2025-03-01'), pd.to_datetime('2025-05-01'))

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2025-05-01':'2025-07-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[2], legend=False)
rme_cleaned_uncorrected.loc['2025-05-01':'2025-07-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[2], legend=False)
ax[2].set_xlim(pd.to_datetime('2025-05-01'), pd.to_datetime('2025-07-01'))

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2025-07-01':'2025-09-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[3], legend=False)
rme_cleaned_uncorrected.loc['2025-07-01':'2025-09-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[3], legend=False)
ax[3].set_xlim(pd.to_datetime('2025-07-01'), pd.to_datetime('2025-09-01'))

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2025-09-01':'2025-11-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[4], legend=False)
rme_cleaned_uncorrected.loc['2025-09-01':'2025-11-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[4], legend=False)
ax[4].set_xlim(pd.to_datetime('2025-09-01'), pd.to_datetime('2025-11-01'))

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2025-11-01':'2026-01-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[5], legend=False)
rme_cleaned_uncorrected.loc['2025-11-01':'2026-01-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[5], legend=False)
ax[5].set_xlim(pd.to_datetime('2025-11-01'), pd.to_datetime('2026-01-01'))

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2025-11-01':'2026-01-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[5], legend=False)
rme_cleaned_uncorrected.loc['2026-01-01':'2026-03-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[6], legend=False)
ax[6].set_xlim(pd.to_datetime('2026-01-01'), pd.to_datetime('2026-03-01'))

#rme_cleaned[rme_cleaned_uncorrected[220.00] < 50].loc['2026-03-01':'2026-05-01'].plot(y=['second_derivative_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax[5], legend=False)
rme_cleaned_uncorrected.loc['2026-03-01':'2026-05-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[7], legend=False)
ax[7].set_xlim(pd.to_datetime('2026-03-01'), pd.to_datetime('2026-05-01'))

rme_cleaned_uncorrected.loc['2026-05-01':'2026-07-01'].plot(y=[220.00], ylabel='Absorbance', ax=ax[8], legend=False)
ax[8].set_xlim(pd.to_datetime('2026-05-01'), pd.to_datetime('2026-07-01'))

for a in ax:
    a.set_xlabel(None)
    a.axhline(45, linestyle='--', color='r')
    a.tick_params(axis='x', rotation=0)
    a.yaxis.set_label_coords(-0.1, 0.5)
    for label in a.get_xticklabels():
        label.set_ha('center')
    
fig.suptitle('RME Absorbance at 220nm')

fig.tight_layout()


### Replotting the whole time series of cleaned values:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_cleaned.plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
rme_cleaned.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_cleaned['2/22/2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
rme_cleaned['2/22/2025':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_cleaned['2025-06-08 00:00':'2025-06-09 00:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#rme_cleaned['2025-05-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'])
#rme['2025-05-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

#rme_cleaned['2025-05-01':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')

rme_cleaned['2025-05-01':].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'])
rme['2025-05-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])


- removing field visit bad data didn't fix all the weird spikes
- weird spikes in nitrate readings correspond to absorbances exceeding ~40. filter those out.
- tried filtering out rows thvalues above 40 

# Filter Outliers

In [ ]:
rme_cleaned[nitrate_cols]

In [ ]:
filtered_nitrate = rme_cleaned[nitrate_cols]
filtered_nitrate = rme_cleaned[nitrate_cols].apply(lambda x: hampel(x,window_size=10, n_sigma = 1.0).filtered_data, axis=0)
filtered_nitrate.index = rme_cleaned.index
rme_filtered = rme_cleaned
#rme_filtered.loc[:, nitrate_cols] = filtered_nitrate


In [ ]:
filtered_nitrate_uncorrected = rme_cleaned_uncorrected[nitrate_cols].apply(lambda x: hampel(x,window_size=10, n_sigma = 1.0).filtered_data, axis=0)
filtered_nitrate_uncorrected.index = rme_cleaned_uncorrected.index
rme_filtered_uncorrected = rme_cleaned_uncorrected
#rme_filtered_uncorrected.loc[:, nitrate_cols] = filtered_nitrate_uncorrected


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_filtered['01/23/2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#rme['02/23/2025':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
#rme_cleaned['2/22/2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_filtered['07/06/2025 00:00':'07/07/2025 12:00'].plot(y=['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Thunderstorm July 6 2025', ax=ax, ylabel='Nitrate (mg/L)')
rme['07/06/2025 00:00':'07/07/2025 12:00'].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
#rme_cleaned['2/22/2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_cleaned['2025-06-08 00:00':'2025-06-09 00:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#rme_cleaned['2025-05-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'])
#rme['2025-05-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])


# Calculate Smoothed and Derivatives

In [ ]:
uncorrected_scan_filtered = scan_helpers.calc_savgol_derivative(rme_cleaned_uncorrected[np.arange(220.0, 735, 2.5)], deriv=0)
uncorrected_scan_1d = scan_helpers.calc_savgol_derivative(rme_cleaned_uncorrected[np.arange(220.0, 735, 2.5)], deriv=1)
uncorrected_scan_2d = scan_helpers.calc_savgol_derivative(rme_cleaned_uncorrected[np.arange(220.0, 735, 2.5)], deriv=2)
uncorrected_scan_2d

# Plotting Feb 2026 Data To make sure insert swap good

In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))
valid_abs = rme_cleaned_uncorrected[220.00] < 40.0

#rme_cleaned.loc[valid_abs]['2026-01-08':].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
#rme_cleaned_uncorrected['2026-01-08':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
rme_cleaned[valid_abs]['2026-01-08':].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Nitrate (mg/L)', ax=ax, label = ['Corrected 2nd Derivative','Corrected Two Wavelength'])


- nice, nitrate and absorbance data look pretty continuous from S::CAN swap
- nitrate data look noisier after the swap, both second derivative and two wavelength. 
- 2nd derivative is way noisier after the swap, to the pont of being unusable. I think its highly advisable to change the inserts.

In [ ]:
fig, ax = plt.subplots(figsize=(11,8.5))
rme_cleaned_uncorrected['2026-02-06':].plot(y=[220.00, 265.00], ylabel='Absorbance', ax=ax, label = ['Corrected 220nm','Corrected 265nm'], secondary_y=True)
# rme_cleaned_uncorrected['2026-01-01':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax, label = ['Uncorrected 220nm','Uncorrected 265nm'])
rme_cleaned['2026-02-06':].plot(y=['second_derivative_no3_mgl_correct', 'two_wavelength_no3_mgl_correct'], ylabel='Absorbance', ax=ax, label = ['Corrected Second Derivative','Corrected Two Wavelength'])


- I see a little nitrate bump Feb 24 2026, is tat the ROS?

In [ ]:
rme_cleaned.loc['2026-02-01 10:00':'2026-02-01 11:00']

## TODO:
- fill missing data from the 5/12-5/15 somehow
- figure out why you're getting negative values of two_wavelength_no3 

# Export

In [ ]:
rme_filtered.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv')
rme_filtered_uncorrected.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected.csv')
uncorrected_scan_filtered.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected_smooth.csv')
uncorrected_scan_1d.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected_1d.csv')
uncorrected_scan_2d.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned_uncorrected_2d.csv')